In [46]:
from datetime import datetime
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from datetime import datetime
import os
from pathlib import Path
import fsspec

import re
pd.set_option('display.max_columns', None)
sys.path.insert(0, os.path.abspath('../..'))
import repo_paths  # noqa: F401
%load_ext autoreload
%autoreload 2
from loading_runs import load_results_from_pickle

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [67]:
norm_models = load_results_from_pickle(folder='ICML_experiments/synthetic/time_checks',
                                        time_threshold='25_09_03_12_00_00',#"25_07_08_14_00_00",
                                        keep_run_type=None,
                                          is_output=False#'bigtestuse_marginal_True'
                                        )#'after_hadi_conversation3')# #"%y_%m_%d_%H_%M_%S"

In [68]:
temp = norm_models.copy()#.drop(['run_type','time_run'])
temp['run_id'] = temp.apply(lambda x: str(x['time_run']) + '_' + str(x['run_type']),axis=1)
temp['model'] = temp['model_type']
temp = temp.query('syn != "Syn7"')


In [69]:
# temp['run_id'] = temp.apply(lambda x: str(x['time_run']) + '_' + str(x['run_type']),axis=1)
# temp['model'] = temp['run_type'].str.extract(r'^(.*?)(?=_[0-9])')

temp['time_run'] = temp[['time_run']].applymap(lambda x: datetime.strptime(x, "%Y-%m-%d_%H-%M-%S"))
temp['time_end'] = temp[['time_end']].applymap(lambda x: datetime.strptime(x, "%Y-%m-%d_%H-%M-%S"))
temp['time_taken'] = temp['time_end'] - temp['time_run']

temp = temp.query('syn != "Syn7"')

In [41]:
# Aggregate runtime statistics (Timedelta)
summary = temp.groupby('model')['time_taken'].agg(
    min='min',
    q10=lambda x: x.quantile(0.10),
    q25=lambda x: x.quantile(0.25),
    q50=lambda x: x.quantile(0.50),  # median
    q75=lambda x: x.quantile(0.75),
    q90=lambda x: x.quantile(0.90),
    max='max',
)

# Helper: Timedelta -> hh:mm:ss
def td_to_hms(td: pd.Timedelta) -> str:
    total_seconds = int(td.total_seconds())
    h, rem = divmod(total_seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

# Format table for display
summary_hms = summary.applymap(td_to_hms)

# Optional: nicer column names
summary_hms = summary_hms.rename(columns={
    'min': 'Min',
    'q10': 'P10',
    'q25': 'P25',
    'q50': 'Median',
    'q75': 'P75',
    'q90': 'P90',
    'max': 'Max',
})

summary_hms.T

model,hide_and_seek,invase,l2x,lasso,lime,random_forest,realx,shap_xgboost
Min,00:00:04,01:18:52,00:00:02,00:00:00,00:14:16,00:00:02,00:01:19,00:00:02
P10,00:00:04,01:19:44,00:00:02,00:00:00,00:23:32,00:00:02,00:01:22,00:00:02
P25,00:00:04,01:21:33,00:00:03,00:00:00,00:27:08,00:00:02,00:01:24,00:00:02
Median,00:00:05,01:25:42,00:00:03,00:00:00,00:30:24,00:00:02,00:01:26,00:00:03
P75,00:00:05,01:37:38,00:00:03,00:00:01,00:32:23,00:00:03,00:01:29,00:00:03
P90,00:00:05,01:51:12,00:00:03,00:00:01,00:36:18,00:00:03,00:01:32,00:00:04
Max,00:00:17,02:06:38,00:00:05,00:00:06,00:39:06,00:00:03,00:01:57,00:00:04


In these runs, I sometimes had multiple tasks running at the same time, which would have slowed things. As such, I am comparing with times of other runs. Where there is a discrepancy, I will report the minimum time in the paper, informed by the quantiles above.

Note, the above table is a print of some older runs. In the newer runs (if you load again), hide and seek and shap we're slower - but again that was because run with other settings.

Ran shap again on better setting and got this. This is for 5 depth. So keeping with 2 seconds.

model,shap_xgboost
Min,00:00:01
P10,00:00:02
P25,00:00:02
Median,00:00:02
P75,00:00:02
P90,00:00:02
Max,00:00:03


So, long story short, will report using the old table.

## 1_000_000 training

In [104]:
from datetime import datetime
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from datetime import datetime
import os
from pathlib import Path
import fsspec

import re
pd.set_option('display.max_columns', None)
sys.path.insert(0, os.path.abspath('../..'))
import repo_paths  # noqa: F401
%load_ext autoreload
%autoreload 2
from loading_runs import load_results_from_pickle, compute_rowwise_metrics, plot_mask_distributions

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [105]:
def compute_rowwise_metrics(row, just_f1=True, bin_mask_column='binary_mask'):
    g = np.array(row['g_test'])        # shape (n, p)
    pred = np.array(row[bin_mask_column])
    
    # True Positives, False Positives, False Negatives per row
    TP = np.sum((pred == 1) & (g == 1), axis=1)
    FP = np.sum((pred == 1) & (g == 0), axis=1)
    FN = np.sum((pred == 0) & (g == 1), axis=1)
    
    # Compute metrics per row
    TPR = TP / (TP + FN + 1e-10)
    FDR = FP / (TP + FP + 1e-10)
    F1  = 2 * TP / (2 * TP + FP + FN + 1e-10)
    
    # Take mean across all rows in this experiment
    if just_f1 == True:
        return pd.Series({
                'F1' : np.mean(F1)
            })
    else:
        return pd.Series({
                'TPR': np.mean(TPR),
                'FDR': np.mean(FDR),
                'F1' : np.mean(F1)
            })

## examining realx

In [128]:
switch = load_results_from_pickle(folder='ICML_experiments/1e6/realx_multiple',
                                        time_threshold='25_09_03_12_00_00',#"25_07_08_14_00_00",
                                        keep_run_type=None,
                                          is_output=False,
                                  numeric_cols = ['lmbda','accuracy', 'roc_auc', 'pct_sig']
                                        )#'after_hadi_conversation3')# #"%y_%m_%d_%H_%M_%S"

temp = switch.copy()

temp['F1'] = temp.apply(compute_rowwise_metrics, axis=1)

temp = temp.query('syn != "Syn7"')

In [129]:
temp.model.value_counts()

REAL-x    54
Name: model, dtype: int64

In [130]:
def find_switch_accuracy(row):
    switch_accuracy = (row['binary_mask'][:,-1] == row['g_test'][:,-1]).mean()
    return switch_accuracy

In [131]:
temp['switch_accuracy'] = temp.apply(lambda row: find_switch_accuracy(row),axis=1)
temp['syn'] = temp.syn.str[-1].astype(int)
# temp['model'] = temp['model'].replace(rename_dict)
cols_to_num = ['TPR_mean', 'FDR_mean', 'f1', 'switch_accuracy', 'seed', 'lmbda']

for col in cols_to_num:
    temp[col] = pd.to_numeric(temp[col])

In [132]:
a = temp[['model','lmbda','syn','TPR_mean', 'FDR_mean','pct_sig','f1']]
a

,model,lmbda,syn,TPR_mean,FDR_mean,pct_sig,f1
0,REAL-x,0.150,1,37.680000,0.000000,0.068509,37.796667
1,REAL-x,0.150,2,58.987500,0.000000,0.214500,71.838381
2,REAL-x,0.150,3,56.865000,0.000000,0.206782,69.724952
3,REAL-x,0.150,4,74.354000,0.215238,0.294991,76.255237
4,REAL-x,0.150,5,57.355333,0.240667,0.220573,64.766992
5,REAL-x,0.150,6,68.910000,0.948333,0.316564,79.610985
6,REAL-x,0.020,1,100.000000,0.000000,0.181818,100.000000
7,REAL-x,0.020,2,67.602500,0.000000,0.245827,77.708762
8,REAL-x,0.020,3,95.262500,0.000000,0.346409,97.292857
9,REAL-x,0.020,4,91.262666,4.274857,0.357000,91.391223


In [133]:
a.groupby(['lmbda'])['f1'].mean()

lmbda
0.005    90.367174
0.020    92.140267
0.040    92.735332
0.060    92.004018
0.080    80.606521
0.100    76.775538
0.120    70.595695
0.140    69.178805
0.150    66.665536
Name: f1, dtype: float64

In [135]:
print(a.query('lmbda == 0.04')[['syn','TPR_mean','FDR_mean']].astype(float).round(0).astype(int))

    syn  TPR_mean  FDR_mean
12    1       100         0
13    2        67         0
14    3        91         0
15    4       100         5
16    5        88         0
17    6        91         4


## comparing

In [143]:
switch = load_results_from_pickle(folder='ICML_experiments/1e6',
                                        time_threshold='25_09_03_12_00_00',#"25_07_08_14_00_00",
                                        keep_run_type=None,
                                          is_output=False,
                                  numeric_cols = ['lmbda','accuracy', 'roc_auc', 'pct_sig']
                                        )#'after_hadi_conversation3')# #"%y_%m_%d_%H_%M_%S"

temp = switch.copy()

temp['F1'] = temp.apply(compute_rowwise_metrics, axis=1)

temp = temp.query('syn != "Syn7"')

In [144]:
temp.model.value_counts()

INVASE       6
LIME         6
L2X          6
Hide&Seek    6
REAL-x       6
SHAP         6
Name: model, dtype: int64

In [145]:
def find_switch_accuracy(row):
    switch_accuracy = (row['binary_mask'][:,-1] == row['g_test'][:,-1]).mean()
    return switch_accuracy

In [146]:
temp['switch_accuracy'] = temp.apply(lambda row: find_switch_accuracy(row),axis=1)
temp['syn'] = temp.syn.str[-1].astype(int)
# temp['model'] = temp['model'].replace(rename_dict)
cols_to_num = ['TPR_mean', 'FDR_mean', 'f1', 'switch_accuracy', 'seed', 'lmbda']

for col in cols_to_num:
    temp[col] = pd.to_numeric(temp[col])

In [147]:
independent_medians = temp.groupby(['model','syn'])[['TPR_mean', 'FDR_mean']].median().reset_index()

In [148]:
independent_medians

,model,syn,TPR_mean,FDR_mean
0,Hide&Seek,1,99.995000,0.000000
1,Hide&Seek,2,100.000000,0.000000
2,Hide&Seek,3,99.442500,0.000000
3,Hide&Seek,4,99.622000,0.840548
4,Hide&Seek,5,97.651333,1.320667
5,Hide&Seek,6,98.274000,1.387643
6,INVASE,1,100.000000,0.000000
7,INVASE,2,100.000000,0.000000
8,INVASE,3,100.000000,0.000000
9,INVASE,4,89.579333,1.262333


In [149]:
# --- start from your medians df ---
df = independent_medians.copy()
df["syn"] = df["syn"].astype(int)

# Round values
df["TPR"] = df["TPR_mean"].round().astype(int)
df["FDR"] = df["FDR_mean"].round().astype(int)

# New model order (matches your NEW TABLE)
model_order = ["Hide&Seek", "INVASE", "REAL-x", "SHAP", "LIME", "L2X"]

# Pivot so rows are syn and columns are model (new format)
pivot_tpr = df.pivot(index="syn", columns="model", values="TPR").reindex(columns=model_order)
pivot_fdr = df.pivot(index="syn", columns="model", values="FDR").reindex(columns=model_order)

# Per-syn bests for bolding
max_tpr_per_syn = pivot_tpr.max(axis=1, skipna=True)
min_fdr_per_syn = pivot_fdr.min(axis=1, skipna=True)

def latex_escape_model(name: str) -> str:
    return name.replace("&", r"\&")

# ----- Header (tabular only, as in your NEW TABLE) -----
colspec = "l|" + "|".join(["cc"] * (len(model_order) - 1)) + "|cc"  # last group no trailing |
header1 = (
    r"\text{Model} "
    + "\n& " + "\n& ".join([rf"\multicolumn{{2}}{{c{'|' if i < len(model_order)-1 else ''}}}{{{latex_escape_model(m)}}}"
                           for i, m in enumerate(model_order)])
    + r" \\"
)

header2 = (
    r"% \text{Metrics}"
    + "\n& " + " & ".join(["TPR & FDR"] * len(model_order))
    + r" \\"
)

latex_rows = []
for syn in range(1, 7):
    row = [f"Syn{syn}"]
    for m in model_order:
        tpr = pivot_tpr.loc[syn, m]
        fdr = pivot_fdr.loc[syn, m]

        # Handle missing gracefully
        if pd.isna(tpr):
            tpr_str = "--"
        else:
            tpr = int(tpr)
            tpr_str = rf"\textbf{{{tpr}}}" if tpr == max_tpr_per_syn.loc[syn] else f"{tpr}"

        if pd.isna(fdr):
            fdr_str = "--"
        else:
            fdr = int(fdr)
            fdr_str = rf"\textbf{{{fdr}}}" if fdr == min_fdr_per_syn.loc[syn] else f"{fdr}"

        row.extend([tpr_str, fdr_str])

    latex_rows.append(" & ".join(row) + r" \\")

latex_table = "\n".join([
    rf"\begin{{tabular}}{{{colspec}}}",
    r"\hline",
    header1,
    header2,
    r"\hline",
    *latex_rows,
    r"\hline",
    r"\end{tabular}",
])

print(latex_table)


\begin{tabular}{l|cc|cc|cc|cc|cc|cc}
\hline
\text{Model} 
& \multicolumn{2}{c|}{Hide\&Seek}
& \multicolumn{2}{c|}{INVASE}
& \multicolumn{2}{c|}{REAL-x}
& \multicolumn{2}{c|}{SHAP}
& \multicolumn{2}{c|}{LIME}
& \multicolumn{2}{c}{L2X} \\
% \text{Metrics}
& TPR & FDR & TPR & FDR & TPR & FDR & TPR & FDR & TPR & FDR & TPR & FDR \\
\hline
Syn1 & \textbf{100} & \textbf{0} & \textbf{100} & \textbf{0} & \textbf{100} & \textbf{0} & 98 & 2 & 24 & 76 & \textbf{100} & \textbf{0} \\
Syn2 & \textbf{100} & \textbf{0} & \textbf{100} & \textbf{0} & 67 & \textbf{0} & \textbf{100} & \textbf{0} & \textbf{100} & \textbf{0} & \textbf{100} & \textbf{0} \\
Syn3 & 99 & \textbf{0} & \textbf{100} & \textbf{0} & 91 & \textbf{0} & \textbf{100} & \textbf{0} & 98 & 2 & 85 & 15 \\
Syn4 & \textbf{100} & \textbf{1} & 90 & \textbf{1} & \textbf{100} & 5 & 70 & 38 & 55 & 49 & 83 & 32 \\
Syn5 & \textbf{98} & 1 & 84 & 1 & 88 & \textbf{0} & 73 & 36 & 50 & 53 & 90 & 29 \\
Syn6 & \textbf{98} & \textbf{1} & 90 & \textbf{1} & 91

## zz_old 1_000_000 training - times and results

below is old not yet run for norm

In [3]:
others2 = load_results_from_pickle(folder='AI_STATS_syn_others_1e6',
                                        time_threshold='25_08_03_00_00_00',#"25_07_08_14_00_00",
                                        keep_run_type=None,
                                          is_output=False#'bigtestuse_marginal_True'
                                        )#'after_hadi_conversation3')# #"%y_%m_%d_%H_%M_%S"
temp = others2.copy()#.drop(['run_type','time_run'])

runs = temp.loc['time_run'].value_counts().index

In [6]:
for run in runs:
    df_times = []
    expy = temp.iloc[:,(temp.loc['time_run']== run).values]
    assert expy.shape[1]==7
    print(expy.loc['run_type'].iloc[0])
    expy.columns = expy.iloc[0]  # Set the columns to the first row
    expy = expy[1:]              # Drop the first row
    expy = expy.loc[['time_run','time_end']].drop('Syn7', axis=1)
    expy = expy.applymap(lambda x: datetime.strptime(x, "%Y-%m-%d_%H-%M-%S"))
    diffs = expy.loc["time_end"] - expy.loc["time_run"]
    df_times.append(diffs)

    arr_sec = ((np.sum(df_times, axis=0)/len(df_times)).astype("timedelta64[s]").mean()).astype(int)
    hours   = arr_sec // 3600
    minutes = (arr_sec % 3600) // 60
    seconds = arr_sec % 60
    print(f"{hours:02d}:{minutes:02d}:{seconds:02d}")
    print("")

l2x_500_0.6_None
00:00:33

shap_xgboost_500_0.6_None
00:01:26

invase_10000_0.1_1000
01:39:06

lime_500_0.6_None
00:10:44

lasso_500_0.6_None
00:00:05

random_forest_500_0.6_None
00:03:46

invase_10000_0.1_1000
01:38:49

invase_10000_0.1_1000
01:48:05

hide_and_seek_500_0.3_None_seed_0
00:04:47



In [7]:
expys = []
for run in runs:
    expy = temp.iloc[:,(temp.loc['time_run']== run).values]
    assert expy.shape[1]==7
    print(expy.loc['run_type'].iloc[0])
    expy.columns = expy.iloc[0]  # Set the columns to the first row
    expy = expy[1:]              # Drop the first row
    # expy = expy.loc[['TPR_mean','FDR_mean','run_type']].drop('Syn7', axis=1)
    for col in expy:
        print(col)
        print('TPR_mean', expy.loc['TPR_mean', col])
        print('FDR_mean', expy.loc['FDR_mean', col])
        print()
    expys.append(expy)

l2x_500_0.6_None
Syn1
TPR_mean 99.99999950000003
FDR_mean 0.0

Syn2
TPR_mean 99.99999975
FDR_mean 0.0

Syn3
TPR_mean 86.37199978407
FDR_mean 13.627999965930002

Syn4
TPR_mean 90.16113309655195
FDR_mean 26.776599946446805

Syn5
TPR_mean 91.64346641745483
FDR_mean 28.133999943732004

Syn6
TPR_mean 87.6841998246316
FDR_mean 12.315799975368403

Syn7
TPR_mean 77.22416649304473
FDR_mean 31.758833280401948

shap_xgboost_500_0.6_None
Syn1
TPR_mean 98.27999950860003
FDR_mean 1.7199999914

Syn2
TPR_mean 99.99999974999999
FDR_mean 0.0

Syn3
TPR_mean 99.99999974999999
FDR_mean 0.0

Syn4
TPR_mean 67.13066651044979
FDR_mean 39.455999921087994

Syn5
TPR_mean 67.24666651007111
FDR_mean 39.383999921232

Syn6
TPR_mean 75.17999984964001
FDR_mean 24.819999950360007

Syn7
TPR_mean 82.02333312806113
FDR_mean 31.689999947183335

invase_10000_0.1_1000
Syn1
TPR_mean 99.99999950000003
FDR_mean 0.0

Syn2
TPR_mean 99.99999974999999
FDR_mean 0.0

Syn3
TPR_mean 89.91249977521875
FDR_mean 0.0

Syn4
TPR_mean 89.37599

In [53]:
expys[3]

syn,Syn1,Syn2,Syn3,Syn4,Syn5,Syn6,Syn7
TPR_mean,100.0,100.0,89.9125,89.376,84.312,90.174,65.91
FDR_mean,0.0,0.0,0.0,1.155,0.887833,0.936627,20.607242
TPR_std,0.0,0.0,12.264984,10.341371,16.63871,9.998486,39.171769
FDR_std,0.0,0.0,0.0,6.918765,6.169668,6.238845,38.899348
g_test,"[[1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[[0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0,...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0,...","[[0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0,...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0,...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0,...","[[1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0,..."
binary_mask,"[[1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[[0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0,...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0,...","[[0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0,...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0,...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0,...","[[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0,..."
batch_size,1000,1000,1000,1000,1000,1000,1000
mask,"[[0.99868965, 0.99845827, 0.0069155204, 0.0080...","[[0.0077792276, 0.008110769, 0.9993014, 0.9992...","[[0.011346302, 0.0092343055, 0.010212383, 0.00...","[[8.368764e-06, 5.7338616e-06, 0.99999946, 0.9...","[[1.1579435e-05, 1.19199385e-05, 0.028613508, ...","[[0.04774443, 0.038253188, 8.293344e-08, 5.514...","[[0.4907503, 0.9999969, 0.00311335, 0.00274206..."
val_predict,"[[0.26972994, 0.7302701], [0.6130104, 0.386989...","[[0.9734535, 0.026546508], [0.14993921, 0.8500...","[[0.037853904, 0.96214604], [0.9532536, 0.0467...","[[0.9263537, 0.073646255], [0.21471712, 0.7852...","[[0.027636234, 0.9723637], [0.96398264, 0.0360...","[[0.057071216, 0.94292873], [0.9689793, 0.0310...","[[0.16921392, 0.83078605], [0.4744004, 0.52559..."
epochs,10000,10000,10000,10000,10000,10000,10000
